# Notebook features

Objectif : maitriser les features d'entree modele avant le train et avant le live.

Regle : une feature modele ne doit pas etre ajoutee directement dans un script train/live. Elle passe d'abord ici : description, unite, normalisation, comportement attendu, correlations, puis promotion dans `shared_decision_features`.

## Contrat modele actuel

Cette table lit le contrat reel du code. Elle est l'autorite pour les datasets et les modeles.

Le notebook ne garde pas d'historique de features retirees : seules les features d'entree modele actuelles doivent apparaitre ici.

In [35]:
from pathlib import Path

import pandas as pd

from shared_decision_features.contract import (
    CATEGORICAL_BY_STAGE,
    FEATURE_FORMULAS,
    FEATURES_BY_STAGE,
)

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 200)

model_features = set().union(*FEATURES_BY_STAGE.values())

In [36]:
rows = []
for stage, features in FEATURES_BY_STAGE.items():
    categorical = set(CATEGORICAL_BY_STAGE.get(stage, ()))
    for name in features:
        rows.append(
            {
                "stage": stage,
                "feature": name,
                "type": "categorielle" if name in categorical else "numerique",
                "obligatoire_modele": True,
                "description": FEATURE_FORMULAS.get(name, "A documenter avant modification."),
            }
        )

contract_df = pd.DataFrame(rows)
contract_df

,stage,feature,type,obligatoire_modele,description
0,preflop,features.hero_position,categorielle,True,"Normalized hero position: SB, BB, BTN, CO, MP, UTG, else UNKNOWN."
1,preflop,features.pot_bb,numerique,True,pot / big_blind.
2,preflop,features.to_call_bb,numerique,True,to_call / big_blind; 0.0 for a free check.
3,preflop,features.effective_stack_bb,numerique,True,"min(hero_stack, max(active_villain_stacks)) / big_blind; folded players excluded."
4,preflop,features.can_check,numerique,True,"1 if check is currently legal, else 0."
5,preflop,features.can_call,numerique,True,"1 if call is legal and to_call_bb > 0, else 0."
6,preflop,features.can_raise,numerique,True,"1 if bet/raise/all-in is legal, else 0."
7,preflop,features.players_active,numerique,True,"current players still in pot, hero included; live = active_opponents + 1 (fallback)."
8,preflop,features.equity_win,numerique,True,Single model equity against active opponents; in live = DecisionInput.equity.
9,preflop,features.call_max_bb,numerique,True,Maximum profitable call in big blinds. In live = call_max / big_blind.


## Dictionnaire des features actuelles

Chaque feature ci-dessous est une feature d'entree modele. Les montants sont normalises en big blinds.

In [37]:
FEATURE_REGISTRY = [
    {
        "feature": "features.to_call_bb",
        "unite": "big blind",
        "description_fr": "Montant que hero doit ajouter pour continuer. Vaut 0 si aucune mise n'est a payer.",
        "valeurs": ">= 0",
    },
    {
        "feature": "features.effective_stack_bb",
        "unite": "big blind",
        "description_fr": "Stack effectif entre hero et les adversaires actifs.",
        "valeurs": ">= 0",
    },
    {
        "feature": "features.players_active",
        "unite": "nombre de joueurs",
        "description_fr": "Nombre de joueurs encore actifs dans le coup, hero inclus.",
        "valeurs": ">= 1",
    },
    {
        "feature": "features.equity_win",
        "unite": "probabilite 0..1",
        "description_fr": "Chance de gagner contre les joueurs actifs.",
        "valeurs": "0..1",
    },
    {
        "feature": "features.equity_win_present",
        "unite": "probabilite 0..1",
        "description_fr": "Chance de gagner conservative ajustee aux joueurs presents a la table.",
        "valeurs": "0..1",
    },
    {
        "feature": "features.call_max_bb",
        "unite": "big blind",
        "description_fr": "Call maximum profitable selon le calcul equity/EV amont.",
        "valeurs": ">= 0",
    },
    {
        "feature": "features.call_margin_bb",
        "unite": "big blind",
        "description_fr": "Marge de call : call_max_bb moins montant a ajouter.",
        "valeurs": "nombre reel",
    },
    {
        "feature": "features.pot_certain_bb",
        "unite": "big blind",
        "description_fr": "Pot garanti si hero continue maintenant.",
        "valeurs": ">= 0",
    },
    {
        "feature": "features.pot_probable_bb",
        "unite": "big blind",
        "description_fr": "Pot probable apres continuation de hero et reaction minimale plausible des joueurs actifs restants.",
        "valeurs": ">= pot_certain_bb",
    },
    {
        "feature": "features.pot_probable_margin_bb",
        "unite": "big blind",
        "description_fr": "Pot probable moins ce que hero doit rajouter.",
        "valeurs": "nombre reel",
    },
    {
        "feature": "features.ev_certain_bb",
        "unite": "big blind",
        "description_fr": "Valeur nette avec le pot certain : equity_win fois pot_certain_bb moins ce que hero doit rajouter.",
        "valeurs": "nombre reel",
    },
    {
        "feature": "features.ev_probable_bb",
        "unite": "big blind",
        "description_fr": "Valeur nette avec le pot probable : equity_win fois pot_probable_bb moins ce que hero doit rajouter.",
        "valeurs": "nombre reel",
    },
    {
        "feature": "features.bet_size_bb",
        "unite": "big blind",
        "description_fr": "Sizing disponible pour les streets flop, turn et river.",
        "valeurs": ">= 0",
    },
]

pd.DataFrame(FEATURE_REGISTRY)

,feature,unite,description_fr,valeurs
0,features.to_call_bb,big blind,Montant que hero doit ajouter pour continuer. Vaut 0 si aucune mise n'est a payer.,>= 0
1,features.effective_stack_bb,big blind,Stack effectif entre hero et les adversaires actifs.,>= 0
2,features.players_active,nombre de joueurs,"Nombre de joueurs encore actifs dans le coup, hero inclus.",>= 1
3,features.equity_win,probabilite 0..1,Chance de gagner contre les joueurs actifs.,0..1
4,features.equity_win_present,probabilite 0..1,Chance de gagner conservative ajustee aux joueurs presents a la table.,0..1
5,features.call_max_bb,big blind,Call maximum profitable selon le calcul equity/EV amont.,>= 0
6,features.call_margin_bb,big blind,Marge de call : call_max_bb moins montant a ajouter.,nombre reel
7,features.pot_certain_bb,big blind,Pot garanti si hero continue maintenant.,>= 0
8,features.pot_probable_bb,big blind,Pot probable apres continuation de hero et reaction minimale plausible des joueurs actifs restants.,>= pot_certain_bb
9,features.pot_probable_margin_bb,big blind,Pot probable moins ce que hero doit rajouter.,nombre reel


## Calcul pot certain / pot probable

- `features.pot_certain_bb` : pot visible + ce que hero doit rajouter maintenant.
- `features.pot_probable_bb` : pot certain + contribution minimale plausible des joueurs actifs restants.
- `features.pot_probable_margin_bb` : pot probable moins ce que hero doit rajouter.
- `features.ev_certain_bb` : `equity_win * pot_certain_bb - amount_to_add_bb`.
- `features.ev_probable_bb` : `equity_win * pot_probable_bb - amount_to_add_bb`.

Le calcul exact train/live est porte par `shared_decision_features.train_builder` et `shared_decision_features.live_builder`. Cette cellule sert seulement de lecture humaine.

In [38]:
def derive_pot_features(pot_visible_bb, amount_to_add_bb, players_active, equity_win):
    pot_visible = max(float(pot_visible_bb), 0.0)
    amount_to_add = max(float(amount_to_add_bb), 0.0)
    equity = max(min(float(equity_win), 1.0), 0.0)
    active_players_after_hero = max(int(players_active) - 1, 0)
    pot_certain_bb = pot_visible + amount_to_add
    pot_probable_bb = pot_certain_bb + active_players_after_hero * amount_to_add
    return {
        "features.pot_certain_bb": pot_certain_bb,
        "features.pot_probable_bb": pot_probable_bb,
        "features.pot_probable_margin_bb": pot_probable_bb - amount_to_add,
        "features.ev_certain_bb": equity * pot_certain_bb - amount_to_add,
        "features.ev_probable_bb": equity * pot_probable_bb - amount_to_add,
    }


examples = pd.DataFrame(
    [
        {"cas": "gratuit", "pot_visible_bb": 3.0, "amount_to_add_bb": 0.0, "players_active": 3, "equity_win": 0.64},
        {"cas": "heads-up", "pot_visible_bb": 6.0, "amount_to_add_bb": 2.0, "players_active": 2, "equity_win": 0.64},
        {"cas": "multiway", "pot_visible_bb": 6.0, "amount_to_add_bb": 2.0, "players_active": 4, "equity_win": 0.64},
    ]
)
derived = examples.apply(
    lambda row: derive_pot_features(row.pot_visible_bb, row.amount_to_add_bb, row.players_active, row.equity_win),
    axis=1,
    result_type="expand",
)
pd.concat([examples, derived], axis=1)

,cas,pot_visible_bb,amount_to_add_bb,players_active,equity_win,features.pot_certain_bb,features.pot_probable_bb,features.pot_probable_margin_bb,features.ev_certain_bb,features.ev_probable_bb
0,gratuit,3.0,0.0,3,0.64,3.0,3.0,3.0,1.92,1.92
1,heads-up,6.0,2.0,2,0.64,8.0,10.0,8.0,3.12,4.40
2,multiway,6.0,2.0,4,0.64,8.0,14.0,12.0,3.12,6.96


## Equity active vs present

`features.equity_win` est la chance de win contre les joueurs actifs. `features.equity_win_present` est une version conservative ajustee si le nombre de joueurs presents est superieur au nombre de joueurs actifs.

In [39]:
def equity_present(equity_win, players_active, players_present=None):
    equity = max(min(float(equity_win), 1.0), 0.0)
    active_players = max(int(players_active), 1)
    present_players = active_players if pd.isna(players_present) else max(int(players_present), active_players)

    active_opponents = max(active_players - 1, 1)
    present_opponents = max(present_players - 1, active_opponents)
    exponent = present_opponents / active_opponents
    return equity ** exponent


equity_examples = pd.DataFrame(
    [
        {"equity_win": 0.64, "players_active": 2, "players_present": 2},
        {"equity_win": 0.64, "players_active": 2, "players_present": 3},
        {"equity_win": 0.64, "players_active": 3, "players_present": 5},
    ]
)
equity_examples.assign(
    features_equity_win_present=lambda frame: frame.apply(
        lambda row: equity_present(
            row["equity_win"],
            row["players_active"],
            row["players_present"],
        ),
        axis=1,
    )
)

,equity_win,players_active,players_present,features_equity_win_present
0,0.64,2,2,0.6400
1,0.64,2,3,0.4096
2,0.64,3,5,0.4096


## Verification dataset

Charge le meilleur dataset slim disponible et verifie que toutes les features du contrat actuel sont presentes.

In [40]:
DATASET_CANDIDATES = [
    Path("outputs/readiness/merged_oracle_3intent_v3_slim/model_input.csv"),
    Path("dist/ml_dataset_export/solver_policy_real_smoke_20/training_smoke/model_input.csv"),
]

dataset_path = next((path for path in DATASET_CANDIDATES if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Aucun dataset modele trouve")

df = pd.read_csv(dataset_path)
dataset_path, df.shape, df.head(3)

(WindowsPath('outputs/readiness/merged_oracle_3intent_v3_slim/model_input.csv'),
 (206145, 20),
   source_dataset  \
 0     PokerBench   
 1     PokerBench   
 2     PokerBench   
 
                                                        source_row_id  hand_id  \
 0   pokerbench:postflop_10k_test_set_game_scenario_information.csv:0      NaN   
 1   pokerbench:postflop_10k_test_set_game_scenario_information.csv:1      NaN   
 2  pokerbench:postflop_10k_test_set_game_scenario_information.csv:10      NaN   
 
   stage_group metadata.street  split  features.bet_size_bb  \
 0    POSTFLOP           RIVER  train                   0.0   
 1    POSTFLOP            TURN  train                   1.0   
 2    POSTFLOP           RIVER   test                   0.0   
 
    features.call_margin_bb  features.call_max_bb  features.effective_stack_bb  \
 0                 2.594595              2.594595                        100.0   
 1                26.068035             27.068035                     

In [41]:
missing_contract_features = sorted(model_features - set(df.columns))
{
    "missing_contract_features": missing_contract_features,
    "feature_count_in_contract": len(model_features),
}

{'missing_contract_features': ['features.can_call',
  'features.can_check',
  'features.can_raise',
  'features.hero_position',
  'features.pot_bb'],
 'feature_count_in_contract': 12}

## Correlations

Les correlations fortes ne sont pas automatiquement mauvaises, mais elles signalent les features redondantes ou a surveiller.

In [42]:
numeric_feature_cols = [feature for feature in sorted(model_features) if feature in df.columns]
numeric = df[numeric_feature_cols].apply(pd.to_numeric, errors="coerce")
corr = numeric.corr(method="spearman")
corr

,features.bet_size_bb,features.call_margin_bb,features.call_max_bb,features.effective_stack_bb,features.equity_win,features.players_active,features.to_call_bb
features.bet_size_bb,1.000000,0.235822,0.301404,-0.045400,0.350044,0.351536,0.667341
features.call_margin_bb,0.235822,1.000000,0.986785,-0.447215,0.700874,0.136853,0.218371
features.call_max_bb,0.301404,0.986785,1.000000,-0.474099,0.717037,0.194908,0.309034
features.effective_stack_bb,-0.045400,-0.447215,-0.474099,1.000000,-0.320701,-0.293687,-0.272056
features.equity_win,0.350044,0.700874,0.717037,-0.320701,1.000000,0.135949,0.296783
features.players_active,0.351536,0.136853,0.194908,-0.293687,0.135949,1.000000,0.625411
features.to_call_bb,0.667341,0.218371,0.309034,-0.272056,0.296783,0.625411,1.000000


In [43]:
strong_pairs = []
for i, left in enumerate(corr.columns):
    for right in corr.columns[i + 1 :]:
        value = corr.loc[left, right]
        if pd.notna(value) and abs(value) >= 0.85:
            strong_pairs.append({"feature_a": left, "feature_b": right, "spearman": value})

pd.DataFrame(strong_pairs).sort_values("spearman", key=lambda s: s.abs(), ascending=False) if strong_pairs else pd.DataFrame(columns=["feature_a", "feature_b", "spearman"])

,feature_a,feature_b,spearman
0,features.call_margin_bb,features.call_max_bb,0.986785
